# 第 17 讲：多模态大模型 (Multimodal Models)

> **核心议题**：从纯文本大语言模型 (Text $\rightarrow$ Text) 演进至跨越文本、图像、音频与视频的统一多模态大模型 (Omni Models)。

- **往期内容**：纯文本语言模型（Token 预测、Transformer 架构、系统优化与后训练对齐）
- **多模态世界**：真实世界的信息是多模态共存的（文本、高分辨率图像、视频流、连续音频信号、3D 场景等）

<img src="images/multimodality.png" width="600" />

### 终极目标：原生全能模型 (Omni Model)
- **通用输入理解**：能够接收任意模态的任意组合输入（图像、视频、音频、纯文本）；
- **通用输出生成**：能够自回归或扩散生成任意模态的任意组合输出。

### 当前技术现状与核心约束
- **Transformer 的普适性**：基于自注意力机制的 Transformer 架构在各类序列建模中表现极其卓越，是现代多模态模型的核心底座；
- **Token 化的世界**：Transformer 依赖 Token 序列进行计算（无论是离散离散 Token 还是连续 Embedding Token），每个 Token 代表一段语义信息单元；
- **核心工程挑战**：文本的分词与 Token 化已非常成熟，但**非文本模态（图像、音频、视频）如何高效、无损且具语义可压缩性地转换为 Token**，是多模态模型最具挑战性的核心课题。

### 本讲两大核心问题：
1. **多模态输入与理解**：如何高效编码非文本数据（如高分辨率图像、长视频）并注入大语言模型？
2. **多模态输出与生成**：如何统一建模并自回归生成连续/离散非文本数据（如直接生成图像或音频）？


## 1. 图像特征对齐与视觉编码器 (Vision Encoders)

### 1.1 CLIP (对比式语言-图像预训练, OpenAI 2021)
[论文链接: CLIP (Radford et al., 2021)](https://arxiv.org/abs/2103.00020)

#### 背景与动机
- **传统计算机视觉**：依赖 ImageNet 等由人工精细标注的封闭分类类别标签，泛化能力受限且难以拓展至开放词汇；
- **核心构想**：能否直接利用互联网上自然存在的数亿海量“（图像, 旁注文本）”对进行无监督/弱监督对齐预训练？

<img src="images/clip.png" width="800" />

#### 对比学习方法 (Contrastive Learning)
- 采样一个超大 Batch 的图像-文本对（例如 $N = 32,768$）；
- 分别通过**图像编码器 (Vision Encoder)** 与**文本编码器 (Text Encoder)** 提取特征向量并做 $L_2$ 归一化；
- 构建 $N \times N$ 的余弦相似度矩阵；
- **双向对齐目标**：
  1. 对每一幅图像，将其在 Batch 内与真实配对文本的余弦相似度最大化（InfoNCE 损失）；
  2. 对每一段文本，将其与真实配对图像的余弦相似度最大化；
  3. 对角线上的 $N$ 个正样本对概率最大化，非对角线上的 $N(N-1)$ 个负样本对概率最小化。

<img src="images/clip-code.png" width="400" />

#### 数据集与数据预处理
- **WIT-400M 数据集**：从全网抓取 4 亿个（图像, 文本）对（OpenAI 未公开发布，后被社区通过 [LAION-5B (Schuhmann et al., 2022)](https://arxiv.org/abs/2212.07143) 与 OpenCLIP 开源复现）；
- **图像预处理流程**：
  1. 真实网页图片分辨率各异（任意长宽比 $W \times H$）；
  2. 使用双三次插值 (Bicubic Interpolation) 将短边等比缩放至 336 像素；
  3. 中心裁剪 (Center Crop) 截取中间的 $336 \times 336$ 区域输入编码器。

#### 视觉与文本编码器架构
- **视觉编码器 (Vision Encoder)**：探索了 ResNet-50 与 [Vision Transformer (ViT, Dosovitskiy et al., 2020)](https://arxiv.org/pdf/2010.11929)；
  - 最终选用 **ViT-L/14@336px**（Large 规模，将图像切分为 $14 \times 14$ 的 Patch，支持 $336 \times 336$ 输入分辨率）；
  - 采用 Attention Pooling 替代简单的平均池化（以全局激活均值作为 Query 进行交叉注意力汇聚）。
- **文本编码器 (Text Encoder)**：63M 参数的 12 层 GPT-2 Transformer 解码器，以最高层的 `[EOS]` Token 特征作为整段文本的全局语义表示。

<img src="images/vit.png" width="600" />

#### 标志性成果与算力效率
- **零样本分类突破**：在 ImageNet-1K 上，未经任何监督微调的 Zero-Shot CLIP 准确率直接超越了在 120 万标注图片上训练的经典有监督 ResNet-50；
- **对比学习 vs 生成式预测**：直接通过图像生成文本的自回归模型在计算效率上远逊于基于对比排序的 CLIP 目标，CLIP 在相同算力下学到了更稠密的语义特征！

<img src="images/clip-efficiency.png" width="400" />

#### CLIP 的局限性
1. 依赖极大的 Batch Size（32K+），跨卡全局 Softmax 通信开销高昂；
2. 图像被粗暴中心裁剪缩放至 $336 \times 336$，丢失了细粒度高分辨率细节（如小文本 OCR 与小物体检测）；
3. 仅对齐了全局图像级语义，缺乏空间局部定位能力。


### 1.2 SigLIP (基于 Sigmoid 损失的图像-语言预训练, Google 2023)
[论文链接: SigLIP (Zhai et al., 2023)](https://arxiv.org/abs/2303.15343)

#### 核心算法革新：Sigmoid 二分类替代全局 Softmax
- **CLIP 的 Softmax 损失**：将一个 Batch 内的所有负样本作为多分类的分母归一化项，导致**损失函数与 Batch Size 深度绑定**，且多卡分布式训练必须进行高开销的 All-Gather 梯度同步通信；
- **SigLIP 的 Sigmoid 损失**：将对比学习重构为独立的**二分类交叉熵 (BCE)** 任务：
  - 对配对的 $(I_i, T_i)$ 正样本对：最大化 $\sigma(z_I \cdot z_T)$；
  - 对不配对的 $(I_i, T_j)$ 负样本对：最小化 $\sigma(z_I \cdot z_T)$。

<img src="images/siglip-code.png" width="500" />

#### 数据集与惊人的计算效率
- **WebLI 数据集**：从全网抓取的 10 亿级图文对，利用自动化 OCR 提取图片文字，仅保留前 10% 极高质量对（支持 100 种多语言）；
- **极致的并行加速**：
  - CLIP：在 256 块 TPUv3 上训练 10 天；
  - SigLIP：在仅 32 块 TPUv4 上训练 5 天即可达到甚至超越 CLIP 的性能，显存占用与计算开销大幅下降！

<img src="images/siglip-parallelism.png" width="800" />

#### 批次解耦特性
- 彻底解耦了 Batch Size 与损失函数计算，在较小 Batch Size（如 16K~32K）下依然能保持极高稳定性，已成为现代视觉语言模型（如 LLaVA OneVision、Qwen3-VL、Gemma-2-VL）事实上的主流视觉编码器底座。


## 2. 视觉语言模型注入范式 (Injecting Visual Tokens into LLMs)

### 2.1 LLaVA (大型语言与视觉助手, 2023)
[论文链接: LLaVA (Liu et al., 2023)](https://arxiv.org/abs/2304.08485)

#### 核心架构模板
1. **视觉编码器**：冻结的 CLIP ViT-L/14；
2. **多模态投影层 (Projector)**：简单的单层线性投影矩阵 $W$（将视觉特征维度映射到大语言模型的词嵌入空间）；
3. **文本解码器**：Vicuna / LLaMA 底座大语言模型。

<img src="images/llava-architecture.png" width="600" />

#### 基于 GPT-4 的多模态指令合成数据
- 利用 MS-COCO 图像已有的边界框 (Bounding Box) 与人工标注 Caption；
- 提示 GPT-4 仅凭结构化文本描述合成出 15.8 万条高质量多轮视觉问答、深度推理与对话数据；

<img src="images/llava-gen.png" width="600" />

#### 两阶段极简训练策略
- **阶段 1（特征对齐 Pre-training）**：冻结视觉编码器和语言模型，仅训练线性映射层 $W$，使视觉 Token 投影到 LLM 词向量空间；
- **阶段 2（端到端指令微调 Visual SFT）**：保持视觉编码器冻结，同时微调投影层 $W$ 和大语言模型参数。

<img src="images/llava-example.png" width="600" />


### 2.2 LLaVA OneVision (全能视觉与 AnyRes 动态分辨率, 2024)
[论文链接: LLaVA-OneVision (Li et al., 2024)](https://arxiv.org/pdf/2408.03326)

#### 核心升级点
- **编码器与底座**：采用 SigLIP 视觉编码器 + 2 层 MLP 投影层 + Qwen-2 72B 语言模型；
- **AnyRes 动态高分辨率切片技术**：
  - 痛点：传统 CLIP 将图片强制缩放至 $336 \times 336$ 会造成图表与 OCR 文本严重模糊；
  - 方案：根据原图宽高比，将图片动态切分为 $a \times b$ 个网格切片（每个切片独立以原生分辨率送入 SigLIP），同时保留一张全局缩略图，拼接后生成丰富的视觉 Token 序列。

<img src="images/llava-onevision-anyres.png" width="600" />

#### 统一单图、多图与视频输入 (Single Image, Multi-Image, Video)
- **单图像**：采用 AnyRes 获得高密度高分辨率 Token；
- **多图对比**：每张图采用基础分辨率切片，控制整体上下文长度；
- **视频序列**：按固定帧率抽帧，每帧采用低分辨率编码，实现模态长度的动态均衡。

<img src="images/llava-onevision-modalities.png" width="600" />

#### 跨模态知识迁移 (Modality Transfer)
- 在单图上学习的图表分析能力可平滑泛化到多图对比；
- 在单图 OCR 与多图关联上训练的模型，能零样本迁移至复杂的操作系统 GUI Agent 交互定位任务中！

<img src="images/llava-onevision-transfer-s1.png" width="600" />
<img src="images/llava-onevision-transfer-s2.png" width="600" />


### 2.3 Qwen-VL 系列演进 (Qwen-VL $\rightarrow$ Qwen2-VL $\rightarrow$ Qwen3-VL)

#### 1. 初代 Qwen-VL (2023)
[论文链接: Qwen-VL (Bai et al., 2023)](https://arxiv.org/abs/2308.12966)
- **视觉编码器**：OpenCLIP ViT-bigC ($14 \times 14$ Patch)；
- **Cross-Attention 压缩适配器**：引入一层交叉注意力层结合 2D 绝对位置编码，将任意分辨率图片固定压缩为 256 个视觉 Token；
- **三阶段递进训练**：
  1. 阶段 1：海量弱标注数据预训练（冻结 LLM，训练视觉编码器与适配器）；
  2. 阶段 2：高质量多任务多分辨率训练（放开全量参数更新）；
  3. 阶段 3：多模态对话与指令对齐（冻结视觉编码器，微调适配器与 LLM）。

<img src="images/qwen-vl-stages.png" width="700" />

---

#### 2. Qwen2-VL: 动态原生分辨率与多模态旋转位置编码 (MRoPE, 2024)
[论文链接: Qwen2-VL (Wang et al., 2024)](https://arxiv.org/abs/2409.12191)

- **原生动态分辨率 (Naive Dynamic Resolution)**：视觉 ViT 直接处理任意长宽比的 Patch 序列，相邻 $2 \times 2$ 的视觉 Token 经 2D 卷积压缩合并为单个 Token，彻底消除几何形变与长宽比失真；
- **多模态旋转位置编码 (Multimodal RoPE / MRoPE)**：
  - 核心突破：将传统 1D 文本位置编码分解为三维独立坐标向量：**时间维度 $t$、垂直高度维度 $h$、水平宽度维度 $w$**；
  - 使得模型能够对长视频时间轴、图像空间拓扑结构与交错文本进行统一的空间几何建模！

<img src="images/qwen2-vl-mrope.png" width="600" />

---

#### 3. Qwen3-VL: 原生全模态与长思维链 (2025/2026)
[论文链接: Qwen3-VL (Alibaba, 2025)](https://arxiv.org/abs/2511.21631)

- **底座规模与长上下文**：支持高达 235B-A22B 的 MoE 稀疏架构，原生支持 256K 超长多模态上下文；
- **视觉编码器与交错 MRoPE (Interleaved MRoPE)**：
  - 采用 SigLIP-2，将多维位置编码按 $[t, w, h, t, w, h]$ 交错分配至高频与低频旋转带；
  - 引入显式视频时间戳 Token（Timestamp Tokens）；
- **跨层特征融合适配器 (DeepStack)**：将视觉编码器不同层级的浅层几何特征与深层语义特征直接跨层注入到大语言模型的多层 Transformer 中；
- **平方根归一化 Token 损失 (Square-root Normalized Loss)**：平衡视频长 Token 与文本短 Token 的梯度贡献，极大提升预训练数值稳定性；
- **长思维链视觉强化学习 (Vision CoT & RLVR)**：在复杂图表几何推导与高难度视觉问答上展现出极高的逻辑推理能力。

<img src="images/qwen3-vl.png" width="700" />
<img src="images/qwen3-vl-results.png" width="600" />


## 3. 原生全离散生成模型 (Towards Omni Models: Chameleon)
[论文链接: Chameleon (Meta, 2024)](https://arxiv.org/pdf/2405.09818)

### 离散 Token 化与统一自回归生成
- **现有 VLM 的固有缺陷**：只能“看”（理解图像），不能“画”（生成图像通常需要外挂独立的 Diffusion 扩散模型，系统割裂）；
- **Chameleon 的核心哲学**：将文本与图像**全部映射为离散的离散 Token 字典**，在同一个 Transformer 内进行统一的自回归下一个 Token 预测！

<img src="images/chameleon.png" width="600" />
<img src="images/chameleon-example.png" width="600" />

### VQ-VAE 离散图像量化
- **矢量量化自编码器 (VQ-VAE)**：
  - 训练一个包含 8192 个码本向量 (Codebook) 的离散量化自编码器；
  - 将 $512 \times 512$ 分辨率的连续图像压缩编码为 $1024$ 个离散 Token；
  - 解码器可直接将离散 Token 序列高质量还原为连续 RGB 图像。

<img src="images/vq-vae.png" width="600" />

### 混合模态训练的稳定性挑战
- **Logit 漂移与范数爆炸 (Logit Drift & Norm Growth)**：
  - 文本 Token 的信息熵较低，而图像离散 Token 的信息熵极高；
  - 模态交叉训练极易引发自注意力层的 Query-Key 范数发散与数值溢出；
- **稳定性解决方案**：引入 **QK-Norm（Query/Key 归一化）** 与 **z-loss 正则化**，保证长序列超大 Batch 训练平稳收敛。

### 总结与未来展望
- **优雅性**：统一了图像理解与生成的自回归范式；
- **代价与权衡**：离散化 Token 必然伴随细粒度局部信息的丢失（在精细 OCR 上稍逊于连续嵌入的 VLM）；
- **行业主流趋势**：连续特征编码器 (SigLIP/ViT) + 大语言模型 (LLM) + 扩散解码头 (Diffusion Head) 构成了目前性能最强悍的工业级 Omni 模型架构。

---

## 本讲核心总结

1. **多模态是大模型的必然演进方向**：理解多模态世界与自主规划行动是通往通用人工智能 (AGI) 的必经之路；
2. **视觉特征对齐两座大山**：CLIP (对比 InfoNCE 损失) 与 SigLIP (解耦 Sigmoid 损失) 奠定了视觉表征的工业基石；
3. **注入范式与动态分辨率**：LLaVA 确立了“Vision Encoder + Projector + LLM”标准模板，AnyRes 与 MRoPE 彻底解决了高分辨率与时空拓扑建模难题；
4. **理解与生成的统一**：从 Chameleon 离散自回归到现代流匹配 (Flow Matching) / 扩散生成，原生多模态生成正在迎来全新爆发！
